# 07 · Features Full Recheck
Independent, full re-extraction of all 6,558 clips in `data/processed/labels.csv` to verify numerical exactness against `data/processed/audio_features.npy`.

In [1]:
# Cell 1 – print sys.executable & import functions
import sys
from pathlib import Path

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

from src.preprocessing.audio import load_fixed, extract_features
print('Successfully imported load_fixed, extract_features from src.preprocessing.audio')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported load_fixed, extract_features from src.preprocessing.audio


In [2]:
# Cell 2 – load labels.csv and audio_features.npy; assert shapes
import datetime
import numpy as np
import pandas as pd

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'

labels = pd.read_csv(labels_path)
X = np.load(features_path)

feat_size = features_path.stat().st_size
feat_mtime = datetime.datetime.fromtimestamp(features_path.stat().st_mtime).isoformat()

print(f'audio_features.npy file size     : {feat_size:,} bytes')
print(f'audio_features.npy last-modified : {feat_mtime}')
print(f'X.shape                          : {X.shape}')
print(f'len(labels)                      : {len(labels):,}')

assert X.shape == (6558, 128), f'Expected X.shape == (6558, 128), got {X.shape}'
assert len(labels) == 6558, f'Expected len(labels) == 6558, got {len(labels)}'
print('Cell 2 asserts passed ✓')

audio_features.npy file size     : 3,357,824 bytes
audio_features.npy last-modified : 2026-11-08T15:43:33.392521
X.shape                          : (6558, 128)
len(labels)                      : 6,558
Cell 2 asserts passed ✓


In [3]:
# Cell 3 – recompute extract_features(load_fixed(path)) for every row into R
import time
from tqdm import tqdm

failed_paths = []
R_list = []

t0 = time.perf_counter()
for idx, row in tqdm(labels.iterrows(), total=len(labels), desc='Recomputing features', mininterval=5.0):
    rel_path = row['path']
    abs_path = REPO_ROOT / rel_path
    try:
        y = load_fixed(abs_path)
        feat = extract_features(y)
        R_list.append(feat)
    except Exception as e:
        failed_paths.append((idx, rel_path, str(e)))

elapsed_s = time.perf_counter() - t0
print(f'Total recomputation time: {elapsed_s:.2f} s ({elapsed_s/60:.2f} min)')
print(f'Number of failures: {len(failed_paths)}')
if len(failed_paths) == 0:
    R = np.stack(R_list).astype(np.float32)
    print(f'Stacked R shape: {R.shape}, dtype: {R.dtype}')

Recomputing features:   0%|          | 0/6558 [00:00<?, ?it/s]

Recomputing features:   0%|          | 1/6558 [00:07<12:54:26,  7.09s/it]

Recomputing features:   4%|▍         | 276/6558 [00:12<03:46, 27.68it/s] 

Recomputing features:   9%|▉         | 584/6558 [00:17<02:22, 41.81it/s]

Recomputing features:  13%|█▎        | 824/6558 [00:22<02:09, 44.11it/s]

Recomputing features:  17%|█▋        | 1145/6558 [00:27<01:46, 51.06it/s]

... (14 outputs left out)

Recomputing features:  91%|█████████▏| 5986/6558 [01:43<00:08, 65.83it/s]

Recomputing features:  97%|█████████▋| 6329/6558 [01:48<00:03, 66.63it/s]

Recomputing features: 100%|██████████| 6558/6558 [01:50<00:00, 59.09it/s]

Total recomputation time: 111.01 s (1.85 min)
Number of failures: 0
Stacked R shape: (6558, 128), dtype: float32


In [4]:
# Cell 4 – assert failures == 0, R.shape == X.shape, and np.allclose
assert len(failed_paths) == 0, f'Recomputation had {len(failed_paths)} failures'
assert R.shape == X.shape, f'Shape mismatch: R is {R.shape}, X is {X.shape}'

# Compute per-row max absolute difference d[i] = max|R[i] - X[i]|
d = np.max(np.abs(R - X), axis=1)

overall_max_d = float(np.max(d))
rows_d_zero = int(np.sum(d == 0.0))
rows_d_gt_tol = int(np.sum(d > 1e-5))

print('=== Alignment & Absolute Difference Summary ===')
print(f'Overall max absolute difference: {overall_max_d:.8e}')
print(f'Rows with d == 0               : {rows_d_zero:,} / {len(d):,}')
print(f'Rows with d > 1e-5             : {rows_d_gt_tol:,} / {len(d):,}')

assert np.allclose(R, X, atol=1e-5), f'np.allclose(R, X, atol=1e-5) failed! Overall max diff is {overall_max_d}'
print('assert np.allclose(R, X, atol=1e-5) PASSED ✓')

=== Alignment & Absolute Difference Summary ===
Overall max absolute difference: 0.00000000e+00
Rows with d == 0               : 6,558 / 6,558
Rows with d > 1e-5             : 0 / 6,558
assert np.allclose(R, X, atol=1e-5) PASSED ✓


In [5]:
# Cell 5 – summary separately for source and app_class
print('=== Breakdown by Source ===')
for src in ['us8k', 'esc50']:
    mask = (labels['source'] == src).values
    d_sub = d[mask]
    max_d_sub = float(np.max(d_sub))
    print(f'Source: {src:<6} | Rows: {len(d_sub):4d} | Max d: {max_d_sub:.8e}')

print('\n=== Breakdown by app_class ===')
for cls_name, grp in labels.groupby('app_class'):
    mask = (labels['app_class'] == cls_name).values
    d_sub = d[mask]
    max_d_sub = float(np.max(d_sub))
    print(f'Class: {cls_name:<18} | Rows: {len(d_sub):4d} | Max d: {max_d_sub:.8e}')

=== Breakdown by Source ===
Source: us8k   | Rows: 6358 | Max d: 0.00000000e+00
Source: esc50  | Rows:  200 | Max d: 0.00000000e+00

=== Breakdown by app_class ===
Class: construction       | Rows: 2080 | Max d: 0.00000000e+00
Class: hvac               | Rows: 1000 | Max d: 0.00000000e+00
Class: music_loudspeaker  | Rows: 1000 | Max d: 0.00000000e+00
Class: siren              | Rows:  969 | Max d: 0.00000000e+00
Class: traffic_horn       | Rows: 1509 | Max d: 0.00000000e+00


In [6]:
# Cell 6 – sensitivity test: shifted pairing must fail
shifted_is_close = np.allclose(R[1:], X[:-1], atol=1e-5)
print(f'Shifted pairing np.allclose(R[1:], X[:-1], atol=1e-5): {shifted_is_close}')
assert not shifted_is_close, 'Sensitivity test failed: shifted pairing unexpectedly evaluated to True!'
print('Sensitivity test PASSED: shifted pairing correctly failed np.allclose ✓')

Shifted pairing np.allclose(R[1:], X[:-1], atol=1e-5): False
Sensitivity test PASSED: shifted pairing correctly failed np.allclose ✓
